In [ ]:
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.mask import mask
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from typing import Tuple, List, Dict
from tqdm import tqdm
import os
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.ndimage import zoom
import numpy as np
import numpy as np
import random
import json
from pathlib import Path
from typing import Dict, List, Tuple
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from typing import Dict, Tuple
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
from pathlib import Path

from data_loader import (
    load_dem, load_infiltration_map, load_landuse_map, print_raster_stats,gmm_no_manila_path_box_load_all_flood_maps,
    load_all_rainfall_scenarios, get_rainfall_stats, visualize_raster, visualize_flood_maps_grid, manila_path_shape_load_all_flood_maps,
    get_raster_stats, load_manila_mask, gmm_path_load_all_flood_maps
)

from data_preprocessing import (normalize_dem, normalize_infiltration, normalize_landuse, handle_nodata )

from flood_maps import (FLOOD_CLASSES, reconstruct_map_from_patches, categorize_all_flood_maps_patch, visualize_all_categorized_maps)

from quadrant import ( stack_spatial_inputs, split_into_quadrants, build_train_test_split, extract_spatial_patches, get_quadrant_patch_indices, visualize_quadrant_split )

from vit import ViT

from training import FloodPatchDataset, train_kfold

RANDOM_SEED = 42
NUM_SCENARIO = 50
PATCH_SIZE = 4

In [ ]:
repo_root = os.getcwd()
gmm_path = os.path.join(repo_root, "COP-30m-GMM")
gmm_no_manila_path = os.path.join(repo_root, "COP-30m-GMM-ManilaMaskOut")
manila_path = os.path.join(repo_root, "COP-30m-ManilaOnly")
rainfall_path = os.path.join(repo_root, "mm_hr_scenarios")

# Loading data

In [ ]:
# Load DEM
print("\n--- Loading DEM ---")
dem_path = os.path.join(gmm_path, "greater_mm_bbox_dem_cop.tif")
if os.path.exists(dem_path):
    dem_data, dem_meta = load_dem(dem_path)
    dem_stats = get_raster_stats(dem_data, "DEM")
    print_raster_stats(dem_stats)
else:
    print(f"WARNING: DEM file not found at {dem_path}")
    dem_data = None

fig1 = visualize_raster(dem_data, title="DEM - Manila", cmap='terrain')

# Load Infiltration Map
print("\n--- Loading Infiltration Map ---")
infilt_path = os.path.join(gmm_path, "GM_Infilt_fixed.tif")
if os.path.exists(infilt_path):
    infilt_data, infilt_meta = load_infiltration_map(infilt_path)
    infilt_stats = get_raster_stats(infilt_data, "Infiltration Map")
    print_raster_stats(infilt_stats)
else:
    print(f"WARNING: Infiltration file not found at {infilt_path}")
    infilt_data = None
fig2 = visualize_raster(infilt_data, title="Infiltration Map", cmap='YlGnBu')

# Load Landuse Map
print("\n--- Loading Landuse Map ---")
landuse_path =  os.path.join(gmm_path, "GM_LU.tif")
if os.path.exists(landuse_path):
    landuse_data, landuse_meta = load_landuse_map(landuse_path)
    landuse_stats = get_raster_stats(landuse_data, "Landuse Map")
    print_raster_stats(landuse_stats)
else:
    print(f"WARNING: Landuse file not found at {landuse_path}")
    landuse_data = None
    
fig3 = visualize_raster(landuse_data, title="Landuse Map", cmap='tab20')

# Load all flood maps
print("\n--- Loading Flood Maps (Ground Truth) ---")
fm_path = os.path.join(gmm_path, "exp-des-4")
flood_maps, flood_metadata = gmm_path_load_all_flood_maps(gmm_path, num_scenarios=NUM_SCENARIO)

if len(flood_maps) > 0:
    print(f"\nFlood maps successfully loaded: {len(flood_maps)}")
    
    if len(flood_maps) >= NUM_SCENARIO:
        flood_stats = get_raster_stats(flood_maps[-1], "Flood Map RS20")
else:
    print("WARNING: No flood maps loaded!")

fig4 = visualize_flood_maps_grid(flood_maps=flood_maps,scenario_ids=list(range(1, 51)),figsize=(25, 20),ncols=5,cmap='terrain')
print_raster_stats(flood_stats)

In [ ]:
print("LOADING RAINFALL SCENARIOS")
# Load rainfall scenarios
rainfall_scenarios = load_all_rainfall_scenarios(rainfall_path)

# Data Preprocessing

In [ ]:
print("\nProcessing DEM...")
dem_clean = handle_nodata(dem_data, nodata_value=dem_meta.get('nodata'), fill_method='mean')

print("\nProcessing Infiltration...")
infilt_clean = handle_nodata(infilt_data, nodata_value=infilt_meta.get('nodata'), fill_method='mean')

print("\nProcessing Landuse...")
landuse_clean = handle_nodata(landuse_data, nodata_value=landuse_meta.get('nodata'), fill_method='interpolate')

In [ ]:
# Normalize DEM
print("\nNormalizing DEM...")
dem_normalized, dem_params = normalize_dem(dem_data, method='minmax')
# Verify normalization
dem_norm_stats = get_raster_stats(dem_normalized, "DEM (Normalized)")
print_raster_stats(dem_norm_stats)

# Normalize Infiltration
print("\nNormalizing Infiltration...")
infilt_normalized, infilt_params = normalize_infiltration(infilt_data)
# Verify normalization
infilt_norm_stats = get_raster_stats(infilt_normalized, "Infiltration (Normalized)")
print_raster_stats(infilt_norm_stats)

# Normalize Landuse
print("\nNormalizing Landuse...")
landuse_normalized, landuse_params = normalize_landuse(landuse_data, method='minmax')
# Verify normalization
landuse_norm_stats = get_raster_stats(landuse_normalized, "Landuse (Normalized)")
print_raster_stats(landuse_norm_stats)


# Flood Maps (Ground Truths)

In [ ]:
flood_maps_categorized_patch, patch_metadata = categorize_all_flood_maps_patch(flood_maps, patch_size=PATCH_SIZE, stride=None, categorization_method='majority', verbose=True)

In [ ]:
reconstructed_maps = []
for patch_labels, metadata in zip(flood_maps_categorized_patch, patch_metadata):
    reconstructed = reconstruct_map_from_patches(patch_labels, metadata, method='nearest')
    reconstructed_maps.append(reconstructed)
fig_all = visualize_all_categorized_maps(reconstructed_maps, scenario_ids=list(range(1, 51)), figsize=(5, 10), ncols=5)
plt.show()

In [ ]:
display(flood_maps_categorized_patch)

# Rainfall

In [ ]:
import numpy as np
import pandas as pd
from typing import Dict, Any, List

# ── Storm Type Registry ───────────────────────────────────────────────────────
NUM_STORM_TYPES = 4
STORM_TYPE_MAP = {
    1: "SCS Front-loaded",
    2: "SCS Balanced",
    3: "SCS Back-loaded",
    4: "Triangular",
}

def _infer_type_id_from_sid(sid: int) -> int:
    """Infer storm type directly from scenario ID (fixed dataset design)."""
    if   1  <= sid <= 10: return 1
    elif 11 <= sid <= 20: return 2
    elif 21 <= sid <= 30: return 3
    elif 31 <= sid <= 50: return 4
    else:
        raise ValueError(f"Scenario ID {sid} out of expected range 1–50")

# ── Loader ────────────────────────────────────────────────────────────────────
def load_scenario_metadata(info_csv_path: str) -> Dict[int, Dict[str, Any]]:
    df = pd.read_csv(info_csv_path)

    required = {
        "No",
        "Pattern Type",
        "Total storm depth P (mm)",
        "Peak time fraction r (tpeak / D)",
    }

    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"info.csv missing columns: {missing}")

    metadata = {}

    for _, row in df.iterrows():
        sid = int(row["No"])
        type_id = _infer_type_id_from_sid(sid)

        # r = 0.0 for SCS types; actual value for triangular
        if type_id in (1, 2, 3):
            r = 0.0
        else:
            r = float(row["Peak time fraction r (tpeak / D)"])

        metadata[sid] = {
            "type_id": type_id,
            "label": STORM_TYPE_MAP[type_id],
            "pattern_type": row["Pattern Type"],
            "total_depth_mm": float(row["Total storm depth P (mm)"]),
            "r": r,
        }

    print(f"Loaded {len(metadata)} scenarios from {info_csv_path}")
    return metadata

# ── One-Hot Encoding ──────────────────────────────────────────────────────────
def get_storm_type_onehot(sid: int, scenario_metadata: Dict[int, Dict[str, Any]]) -> np.ndarray:
    """Return (4,) one-hot vector for storm type."""
    type_id = scenario_metadata[sid]["type_id"]
    onehot = np.zeros(NUM_STORM_TYPES, dtype=np.float32)
    onehot[type_id - 1] = 1.0
    return onehot

def build_all_storm_onehots(scenario_metadata: Dict[int, Dict[str, Any]]) -> np.ndarray:
    """Return (N_scenarios, 4) one-hot matrix."""
    sids = sorted(scenario_metadata.keys())
    onehots = np.stack(
        [get_storm_type_onehot(sid, scenario_metadata) for sid in sids],
        axis=0
    )

    print(f"Storm one-hot matrix shape: {onehots.shape}")
    return onehots

# ── Conditioning Vector ───────────────────────────────────────────────────────
# rain(13) + onehot(4) + r(1) + depth_norm(1) = 19 dims
CONDITIONING_DIM = 19

def build_conditioning_vector(
    rain_normalized: np.ndarray,  # (13,)
    type_id: int,                 # 1
    r: float,                     # 0.0
    total_depth_mm: float,        # 38.0
    depth_min: float,             # 5.0
    depth_max: float,             # 78.0
) -> np.ndarray:
    """Build full (19,) conditioning vector."""

    # Step 1: Create one-hot for storm type
    onehot = np.zeros(4, dtype=np.float32)
    onehot[type_id - 1] = 1.0  # type_id=1 → onehot[0]=1.0
    # Result: [1.0, 0.0, 0.0, 0.0]

    # Step 2: Normalize depth to [0, 1]
    depth_norm = (total_depth_mm - depth_min) / (depth_max - depth_min)

    # Step 3: Concatenate all 19 dimensions
    return np.concatenate([
        rain_normalized.astype(np.float32),   # (13,) rainfall
        onehot,                               # (4,) storm type one-hot
        [np.float32(r)],                      # (1,) r parameter → [0.0]
        [np.float32(depth_norm)],             # (1,) normalized depth → [0.452]
    ])
    # Result: (19,) vector


def build_all_conditioning_vectors(
    rain_embeddings: np.ndarray,  # (50, 13) — normalized rainfall sequences
    scenario_metadata: Dict[int, Dict[str, Any]],  # 50 scenarios' metadata
    depth_min: float,             # Min depth from training (5.0 mm)
    depth_max: float,             # Max depth from training (78.0 mm)
) -> np.ndarray:                  # Returns (50, 19) — full conditioning vectors
    """Build (N_scenarios, 19) conditioning matrix."""

    sids = sorted(scenario_metadata.keys())  # [1, 2, 3, ..., 50]
    vectors = []

    for i, sid in enumerate(sids):
        m = scenario_metadata[sid]  # Get metadata for this scenario

        vec = build_conditioning_vector(
            rain_normalized=rain_embeddings[i],      # (13,) rainfall for scenario i
            type_id=m["type_id"],                    # 1, 2, 3, or 4 (storm type)
            r=m["r"],                                # 0.0 for SCS, tpeak for triangular
            total_depth_mm=m["total_depth_mm"],      # Actual depth (mm)
            depth_min=depth_min,                     # 5.0
            depth_max=depth_max,                     # 78.0
        )

        vectors.append(vec)

    out = np.stack(vectors, axis=0)  # Stack all 50 vectors → (50, 19)

    print(f"Conditioning matrix shape: {out.shape}")
    return out


# ── Validation ────────────────────────────────────────────────────────────────
def validate_and_print_metadata(
    scenario_metadata: Dict[int, Dict[str, Any]],
    train_ids: List[int],
    test_ids: List[int],
) -> None:

    print("\n" + "=" * 70)
    print("SCENARIO METADATA SUMMARY")
    print("=" * 70)

    all_ids = sorted(scenario_metadata.keys())
    print(f"\nTotal scenarios: {len(all_ids)}")

    # Full dataset distribution
    print("\nStorm type distribution (full dataset):")

    for tid, label in STORM_TYPE_MAP.items():
        ids = [sid for sid, m in scenario_metadata.items()
               if m["type_id"] == tid]

        if not ids:
            continue

        depths = [scenario_metadata[sid]["total_depth_mm"] for sid in ids]

        print(f"  Type {tid} - {label:<18}: "
              f"{len(ids):>2} scenarios  "
              f"(IDs {min(ids)}–{max(ids)})  "
              f"depth {min(depths):.0f}–{max(depths):.0f} mm")

    # Train distribution
    print("\nTrain set distribution:")
    for tid, label in STORM_TYPE_MAP.items():
        ids = [sid for sid in train_ids
               if scenario_metadata[sid]["type_id"] == tid]
        print(f"  Type {tid} - {label:<18}: {len(ids)}")

    # Test distribution
    print("\nTest set distribution:")
    for tid, label in STORM_TYPE_MAP.items():
        ids = [sid for sid in test_ids
               if scenario_metadata[sid]["type_id"] == tid]
        print(f"  Type {tid} - {label:<18}: {len(ids)}")

    # Triangular r range
    triangular = [
        m["r"] for m in scenario_metadata.values()
        if m["type_id"] == 4
    ]

    if triangular:
        print(f"\nTriangular r range: "
              f"{min(triangular):.2f} – {max(triangular):.2f}")

    print("=" * 70)

In [ ]:
scenario_metadata = load_scenario_metadata(os.path.join(repo_root, "info.csv"))
validate_and_print_metadata(scenario_metadata, 
                             list(range(1, NUM_SCENARIO+1)),
                             list(range(1, NUM_SCENARIO+1)))

# ── Depth normalization stats (from training data) ────────────────────────────
all_depths = [m['total_depth_mm'] for m in scenario_metadata.values()]
depth_min, depth_max = min(all_depths), max(all_depths)
print(f"Depth range: {depth_min:.1f} – {depth_max:.1f} mm")


In [ ]:
print(scenario_metadata)

# Data Splitting Strategy

In [ ]:
print("Stack Spatial Inputs")
spatial_stack = stack_spatial_inputs(dem_normalized, infilt_normalized, landuse_normalized)

print("Flood maps already categorized →", len(flood_maps_categorized_patch), "scenarios")

print("Quadrant Split — Spatial Stack")
spatial_quadrants = split_into_quadrants(spatial_stack)

print("\nExtract Spatial Patches (full map)")
spatial_patches = extract_spatial_patches(spatial_stack, patch_size=PATCH_SIZE)

print("\nCompute Quadrant Patch Indices")
H, W = dem_normalized.shape
quadrant_indices = get_quadrant_patch_indices(H, W, patch_size=PATCH_SIZE)

print("\nBuild Train / Test Split")
dataset = build_train_test_split(spatial_patches, flood_maps_categorized_patch, quadrant_indices, rainfall_scenarios=rainfall_scenarios,)

In [ ]:
visualize_quadrant_split(dem_normalized)

# Rainfall Encoding

In [ ]:
all_intensities = np.concatenate([df['intensity_mmhr'].values for df in rainfall_scenarios])
rain_min, rain_max = all_intensities.min(), all_intensities.max()

def normalize_rainfall_scenarios(rainfall_scenarios, rain_min, rain_max):
    normalized = []
    for df in rainfall_scenarios:
        intensity = (df['intensity_mmhr'].values - rain_min) / (rain_max - rain_min)
        normalized.append(intensity.astype(np.float32))
    return np.stack(normalized, axis=0) 

rain_embeddings = normalize_rainfall_scenarios(rainfall_scenarios, rain_min, rain_max)
print(rain_embeddings.shape) 

In [ ]:
# ── Build conditioning vectors (19-dim) ───────────────────────────────────────
# rain_embeddings shape: (50, 13) — already built
conditioning_vectors = build_all_conditioning_vectors(
    rain_embeddings   = rain_embeddings,      # (50, 13)
    scenario_metadata = scenario_metadata,
    depth_min         = depth_min,
    depth_max         = depth_max,
)   # → (50, 19)


In [ ]:
print("\nConditioning vector breakdown (dims):")
print("  [0 :13] = normalized rainfall sequence")
print("  [13:17] = storm type one-hot")
print("  [17]    = r  (tpeak fraction)")
print("  [18]    = normalized total depth")

for sid in [1, 11, 21, 31, 40]:
    v = conditioning_vectors[sid - 1]
    print(f"\n  RS{sid:>2} ({scenario_metadata[sid]['label']:<18})"
          f"  rain[:3]={v[:3].round(3)}"
          f"  onehot={v[13:17].astype(int)}"
          f"  r={v[17]:.2f}"
          f"  depth_norm={v[18]:.3f}")

# ViT Config

In [ ]:
from config import Config
from model_config import model_factory, make_model
from training import FloodPatchDataset, train_kfold

cfg = Config()  
cfg.save(cfg.train.output_dir / 'config.json')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dataset = FloodPatchDataset(
    X_spatial = dataset['X_train'],
    y_labels  = dataset['y_train'],
    rainfall  = conditioning_vectors,
)

# Training Setup

In [ ]:
# ── K-Fold Training ─────────────────────────────────────────────────────
fold_histories, best_fold = train_kfold(
    model_factory = model_factory(cfg.model),
    full_dataset  = train_dataset,
    y_train       = dataset['y_train'],
    cfg           = cfg.train,
    device        = device,
)

In [ ]:
# Loading best model
ckpt = torch.load(cfg.train.output_dir / f'fold_{best_fold+1}' / 'best_model.pt', map_location=device)
model = make_model(cfg.model)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()

print(f"✓ Loaded fold {best_fold+1}")
print(f"  Best macro F1: {ckpt.get('best_macro_f1', 'N/A')}")

# Testing

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score, f1_score)
from typing import Dict
from torch.utils.data import DataLoader, TensorDataset

FLOOD_CLASSES = {
    0: {'name': 'No Flood',  'color': '#FFFFFF', 'hex': 'FFFFFF'},
    1: {'name': 'Light',     'color': '#FFEB3B', 'hex': 'FFEB3B'},
    2: {'name': 'Moderate',  'color': '#FF9800', 'hex': 'FF9800'},
    3: {'name': 'Heavy',     'color': '#F44336', 'hex': 'F44336'},
    4: {'name': 'Extreme',   'color': '#9C27B0', 'hex': '9C27B0'},
}
CLASS_NAMES = [FLOOD_CLASSES[i]['name'] for i in range(5)]
FLOOD_CMAP  = ListedColormap([FLOOD_CLASSES[i]['color'] for i in range(5)])
FLOOD_NORM  = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], FLOOD_CMAP.N)

def compute_iou_per_class(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int = 5) -> np.ndarray:
    iou = np.zeros(num_classes, dtype=np.float32)
    for c in range(num_classes):
        tp = ((y_pred == c) & (y_true == c)).sum()
        fp = ((y_pred == c) & (y_true != c)).sum()
        fn = ((y_pred != c) & (y_true == c)).sum()
        denom = tp + fp + fn
        iou[c] = float(tp) / denom if denom > 0 else 0.0
    return iou

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, num_classes: int = 5) -> Dict:
    labels   = list(range(num_classes))
    acc      = accuracy_score(y_true, y_pred)
    prec_pc  = precision_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    rec_pc   = recall_score(   y_true, y_pred, average=None, labels=labels, zero_division=0)
    f1_pc    = f1_score(       y_true, y_pred, average=None, labels=labels, zero_division=0)
    iou_pc   = compute_iou_per_class(y_true, y_pred, num_classes)

    return {
        'accuracy':          acc,
        'precision_macro':   float(prec_pc.mean()),
        'recall_macro':      float(rec_pc.mean()),
        'f1_macro':          float(f1_pc.mean()),
        'iou_macro':         float(iou_pc.mean()),
        'precision_class':   prec_pc,
        'recall_class':      rec_pc,
        'f1_class':          f1_pc,
        'iou_class':         iou_pc,
        'confusion_matrix':  confusion_matrix(y_true, y_pred, labels=labels),
    }

def prepare_ground_truth(y_test: np.ndarray, quadrant_indices: Dict, patch_metadata: Dict) -> Dict:
    print("\nPreparing ground truth maps for test scenarios...")
    N_scenarios, N_test = y_test.shape
    full_n_h = patch_metadata['num_patches_h']
    full_n_w = patch_metadata['num_patches_w']
    n_h = full_n_h - full_n_h // 2
    n_w = full_n_w // 2

    gt_maps = np.zeros((N_scenarios, n_h, n_w), dtype=np.int8)
    for s in range(N_scenarios):
        gt_maps[s] = y_test[s].reshape(n_h, n_w)

    print(f"  Q3 patch grid : {n_h} rows × {n_w} cols = {n_h*n_w} patches")
    print(f"  GT maps shape : {gt_maps.shape}  ✓")
    return {'flat': y_test, 'maps': gt_maps, 'n_h': n_h, 'n_w': n_w}

def predict_scenario(model, X_test, rain_normalized, scenario_idx, batch_size, device):
    model.eval()
    X_t    = torch.tensor(X_test, dtype=torch.float32).permute(0, 3, 1, 2)
    rain_t = (torch.tensor(rain_normalized[scenario_idx], dtype=torch.float32)
              .unsqueeze(0).expand(len(X_test), -1))
    loader = DataLoader(TensorDataset(X_t, rain_t), batch_size=batch_size, shuffle=False)
    preds, confs = [], []
    with torch.no_grad():
        for sp, ra in loader:
            logits, _ = model(sp.to(device), ra.to(device))
            probs     = torch.softmax(logits, dim=1)
            preds.append(probs.argmax(1).cpu().numpy())
            confs.append(probs.max(1).values.cpu().numpy())
    return np.concatenate(preds), np.concatenate(confs)

def evaluate_all_scenarios(model, X_test, y_test, conditioning_vectors, ground_truth, batch_size, device, num_classes) -> Dict:
    print("\nEvaluating all test scenarios...")
    N_scenarios = conditioning_vectors.shape[0]
    n_h, n_w    = ground_truth['n_h'], ground_truth['n_w']
    per_scenario, pred_maps = [], np.zeros((N_scenarios, n_h, n_w), dtype=np.int8)

    conf_maps = np.zeros((N_scenarios, n_h, n_w), dtype=np.float32)   # NEW

    for s in range(N_scenarios):
        preds, confs = predict_scenario(model, X_test, conditioning_vectors, s, batch_size, device)
        labels = ground_truth['flat'][s]
        m      = compute_metrics(labels, preds, num_classes)
        per_scenario.append({'scenario_id': s + 1, **m,
                             'preds_flat': preds, 'confs_flat': confs})   # confs added
        pred_maps[s] = preds.reshape(n_h, n_w)
        conf_maps[s] = confs.reshape(n_h, n_w)                           # NEW
        frac_low = (confs < 0.5).mean() * 100
        print(f"  RS{s+1:02d}  Acc={m['accuracy']:.4f}  Prec={m['precision_macro']:.4f}  "
              f"Rec={m['recall_macro']:.4f}  F1={m['f1_macro']:.4f}  IoU={m['iou_macro']:.4f}"
              f"  Conf={confs.mean():.3f}  Uncertain={frac_low:.1f}%")
        
    def agg(key):
        v = [r[key] for r in per_scenario]
        return float(np.mean(v)), float(np.std(v))

    aggregate = {
        'accuracy':         agg('accuracy'),
        'precision_macro':  agg('precision_macro'),
        'recall_macro':     agg('recall_macro'),
        'f1_macro':         agg('f1_macro'),
        'iou_macro':        agg('iou_macro'),
        'f1_class_avg':     np.mean([r['f1_class']  for r in per_scenario], axis=0),
        'iou_class_avg':    np.mean([r['iou_class'] for r in per_scenario], axis=0),
        'prec_class_avg':   np.mean([r['precision_class'] for r in per_scenario], axis=0),
        'rec_class_avg':    np.mean([r['recall_class']    for r in per_scenario], axis=0),
        'best_scenario':    int(np.argmax([r['f1_macro'] for r in per_scenario])) + 1,
        'worst_scenario':   int(np.argmin([r['f1_macro'] for r in per_scenario])) + 1,
    }

    print("\n" + "="*65)
    print("EVALUATION SUMMARY  —  Q3 Test Region (50 Scenarios)")
    print("="*65)
    for key, label in [('accuracy','Accuracy'), ('precision_macro','Precision (macro)'),
                       ('recall_macro','Recall (macro)'), ('f1_macro','F1 (macro)'),
                       ('iou_macro','IoU (macro)')]:
        m, s = aggregate[key]
        print(f"  {label:22s}: {m:.4f} ± {s:.4f}")
    print(f"  Best  scenario        : RS{aggregate['best_scenario']}")
    print(f"  Worst scenario        : RS{aggregate['worst_scenario']}")
    print("="*65)

    return {
        'per_scenario': per_scenario,
        'aggregate':    aggregate,
        'pred_maps':    pred_maps,
        'conf_maps':    conf_maps,          
        'gt_maps':      ground_truth['maps'],
        'n_h': n_h, 'n_w': n_w,
    }

def plot_metrics_summary(results: Dict) -> plt.Figure:
    agg    = results['aggregate']
    keys   = ['accuracy','precision_macro','recall_macro','f1_macro','iou_macro']
    labels = ['Accuracy','Precision\n(macro)','Recall\n(macro)','F1\n(macro)','IoU\n(macro)']
    colors = ['#2196F3','#4CAF50','#FF9800','#F44336','#9C27B0']
    means  = [agg[k][0] for k in keys]
    stds   = [agg[k][1] for k in keys]

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(keys))
    bars = ax.bar(x, means, yerr=stds, capsize=6, color=colors, alpha=0.85,
                  error_kw={'linewidth':2,'ecolor':'black'}, zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylim(0, 1.12); ax.set_ylabel("Score", fontsize=11)
    ax.set_title("Model Performance — Q3 Test Region  (50 Scenarios, Macro Average)",
                 fontsize=12, fontweight='bold')
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.35)
    ax.grid(axis='y', alpha=0.25, zorder=0)
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.015,
                f'{m:.3f}', ha='center', fontsize=10, fontweight='bold')
    plt.tight_layout()
    return fig

def plot_per_scenario_metrics(results: Dict) -> plt.Figure:
    per  = results['per_scenario']
    scen = [r['scenario_id'] for r in per]
    series = {
        'Accuracy':          [r['accuracy']        for r in per],
        'Precision (macro)': [r['precision_macro']  for r in per],
        'Recall (macro)':    [r['recall_macro']     for r in per],
        'F1 (macro)':        [r['f1_macro']         for r in per],
        'IoU (macro)':       [r['iou_macro']        for r in per],
    }
    colors  = ['#2196F3','#4CAF50','#FF9800','#F44336','#9C27B0']
    markers = ['o','s','^','D','v']

    fig, ax = plt.subplots(figsize=(15, 5))
    for (name, vals), color, marker in zip(series.items(), colors, markers):
        ax.plot(scen, vals, label=name, color=color,
                marker=marker, markersize=3, linewidth=1.5)

    best  = results['aggregate']['best_scenario']
    worst = results['aggregate']['worst_scenario']
    ax.axvline(best,  color='green', linestyle='--', alpha=0.5, label=f'Best (RS{best})')
    ax.axvline(worst, color='red',   linestyle='--', alpha=0.5, label=f'Worst (RS{worst})')
    ax.set_xlabel("Rainfall Scenario", fontsize=11)
    ax.set_ylabel("Score", fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.set_xticks(scen[::5])
    ax.set_xticklabels([f'RS{s}' for s in scen[::5]], rotation=45, fontsize=8)
    ax.set_title("Per-Scenario Metrics — Q3 Test Region", fontsize=13, fontweight='bold')
    ax.legend(fontsize=9, ncol=4); ax.grid(alpha=0.25)
    plt.tight_layout()
    return fig

def plot_per_class_metrics(results: Dict) -> plt.Figure:
    agg  = results['aggregate']
    prec = agg['prec_class_avg']
    rec  = agg['rec_class_avg']
    f1   = agg['f1_class_avg']
    iou  = agg['iou_class_avg']

    x, width = np.arange(5), 0.2
    fig, ax  = plt.subplots(figsize=(11, 5))
    ax.bar(x - 1.5*width, prec, width, label='Precision', color='#4CAF50', alpha=0.85)
    ax.bar(x - 0.5*width, rec,  width, label='Recall',    color='#FF9800', alpha=0.85)
    ax.bar(x + 0.5*width, f1,   width, label='F1',        color='#F44336', alpha=0.85)
    ax.bar(x + 1.5*width, iou,  width, label='IoU',       color='#9C27B0', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, fontsize=11)
    ax.set_ylim(0, 1.05); ax.set_ylabel("Score (avg over 50 scenarios)", fontsize=11)
    ax.set_title("Per-Class Metrics — Q3 Test Region", fontsize=13, fontweight='bold')
    ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.25)
    tick_colors = ['#777777','#B8A000','#CC7700','#C62828','#6A1B9A']
    for tick, c in zip(ax.get_xticklabels(), tick_colors):
        tick.set_color(c); tick.set_fontweight('bold')
    plt.tight_layout()
    return fig

def plot_flood_on_dem(dem_q3: np.ndarray, flood_map: np.ndarray, scenario_id: int, metrics: Dict, mode: str, figsize: tuple, inset_center: tuple, inset_size: int) -> plt.Figure:
    H, W     = dem_q3.shape
    n_h, n_w = flood_map.shape

    # ── Upsample flood map to DEM resolution ─────────────────────────────────
    ph = H / n_h   
    pw = W / n_w  
    flood_full = np.zeros((H, W), dtype=np.float32)
    for i in range(n_h):
        for j in range(n_w):
            r0, r1 = int(i * ph), int((i+1) * ph)
            c0, c1 = int(j * pw), int((j+1) * pw)
            flood_full[r0:r1, c0:c1] = flood_map[i, j]

    # ── RGBA flood overlay ────────────────────────────────────────────────────
    hex_to_rgb = lambda h: tuple(int(h[i:i+2], 16)/255.0 for i in (1,3,5))
    flood_rgba = np.zeros((H, W, 4), dtype=np.float32)
    class_alphas = {0: 0.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0}
    for cls, info in FLOOD_CLASSES.items():
        mask = flood_full == cls
        r, g, b = hex_to_rgb(info['color'])
        flood_rgba[mask, 0] = r
        flood_rgba[mask, 1] = g
        flood_rgba[mask, 2] = b
        flood_rgba[mask, 3] = class_alphas[cls]

    # ── Inset center: default to centroid of highest flood class ─────────────
    if inset_center is None:
        for cls in [4, 3, 2, 1]:
            hi = np.argwhere(flood_map == cls)
            if len(hi) > 0:
                pc = hi[len(hi)//2]
                inset_center = (int(pc[0] * ph + ph/2),
                                int(pc[1] * pw + pw/2))
                break
        if inset_center is None:
            inset_center = (H//2, W//2)

    ic_r, ic_c = inset_center
    iz = inset_size
    ir0 = max(0, ic_r - iz); ir1 = min(H, ic_r + iz)
    ic0 = max(0, ic_c - iz); ic1 = min(W, ic_c + iz)

    # ── DEM display range ─────────────────────────────────────────────────────
    dem_p2, dem_p98 = np.percentile(dem_q3, 2), np.percentile(dem_q3, 98)
    fig = plt.figure(figsize=figsize, facecolor='#1a1a1a')
    ax  = fig.add_axes([0.08, 0.08, 0.72, 0.84], facecolor='#1a1a1a')

    # Layer 1: DEM
    dem_im = ax.imshow(dem_q3, cmap='terrain', vmin=dem_p2, vmax=dem_p98, origin='upper', interpolation='bilinear', alpha=0.4)
    # Layer 2: Flood overlay — fully opaque colors, No Flood transparent
    ax.imshow(flood_rgba, origin='upper', interpolation='nearest')
    # Inset red box marker
    from matplotlib.patches import Rectangle
    rect = Rectangle((ic0, ir0), ic1-ic0, ir1-ir0, linewidth=2, edgecolor='red', facecolor='none', zorder=5)
    ax.add_patch(rect)

    # Axes styling — white text on dark bg
    ax.set_xlabel("Column (West  ->  East)", fontsize=9, color='white')
    ax.set_ylabel("Row (North  ->  South)",  fontsize=9, color='white')
    ax.tick_params(labelsize=8, colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('white')

    mode_label = "Predicted" if mode == 'pred' else "Ground Truth"
    title = f"Metro Manila Q3 -- {mode_label} Flood  (RS{scenario_id})"
    if metrics:
        m = metrics
        title += (f"\nAcc={m['accuracy']:.3f}  Prec={m['precision_macro']:.3f}  "
                  f"Rec={m['recall_macro']:.3f}  F1={m['f1_macro']:.3f}  "
                  f"IoU={m['iou_macro']:.3f}")
    ax.set_title(title, fontsize=11, fontweight='bold', pad=10, color='white')

    # ── DEM colorbar ─────────────────────────────────────────────────────────
    cbar_ax = fig.add_axes([0.82, 0.35, 0.025, 0.50])
    cbar = fig.colorbar(dem_im, cax=cbar_ax)
    cbar.set_label("Elevation (m)", fontsize=9, color='white')
    cbar.ax.tick_params(labelsize=8, colors='white')
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

    # ── Flood class legend ────────────────────────────────────────────────────
    legend_ax = fig.add_axes([0.81, 0.08, 0.17, 0.22])
    legend_ax.set_facecolor('#1a1a1a')
    legend_ax.axis('off')
    legend_ax.set_title("Flood Class", fontsize=8, fontweight='bold',
                        pad=4, color='white')
    for i, (cls, info) in enumerate(reversed(list(FLOOD_CLASSES.items()))):
        color = info['color'] if cls > 0 else '#555555'   
        legend_ax.add_patch(
            plt.Rectangle((0.0, i * 0.19), 0.22, 0.15,
                          facecolor=color, edgecolor='#aaaaaa', linewidth=0.6,
                          transform=legend_ax.transAxes, clip_on=False)
        )
        legend_ax.text(0.28, i * 0.19 + 0.075, info['name'], transform=legend_ax.transAxes, va='center', fontsize=7.5, color='white') 

    # ── Inset zoom panel ─────────────────────────────────────────────────────
    inset_ax = fig.add_axes([0.44, 0.08, 0.28, 0.28], facecolor='#1a1a1a')
    inset_ax.imshow(dem_q3[ir0:ir1, ic0:ic1], cmap='terrain', vmin=dem_p2, vmax=dem_p98, origin='upper', interpolation='bilinear', alpha=0.4)
    inset_ax.imshow(flood_rgba[ir0:ir1, ic0:ic1], origin='upper', interpolation='nearest')
    inset_ax.set_xticks([]); inset_ax.set_yticks([])
    for spine in inset_ax.spines.values():
        spine.set_edgecolor('red'); spine.set_linewidth(2)
    return fig

def plot_best_worst_dem(dem_q3:   np.ndarray, results:  Dict, figsize:  tuple = (18, 11) ) -> plt.Figure:
    best_idx  = results['aggregate']['best_scenario']  - 1
    worst_idx = results['aggregate']['worst_scenario'] - 1
    n_h, n_w  = results['n_h'], results['n_w']

    fig, axes = plt.subplots(2, 2, figsize=figsize)
    fig.suptitle("Best vs Worst Scenario — Q3 DEM + Flood Overlay", fontsize=14, fontweight='bold', y=1.01)

    H, W = dem_q3.shape
    ph, pw = H / n_h, W / n_w
    hex_to_rgb = lambda h: tuple(int(h[i:i+2], 16)/255.0 for i in (1,3,5))
    class_alphas = {0:0.0, 1:0.55, 2:0.65, 3:0.75, 4:0.88}
    dem_p2, dem_p98 = np.percentile(dem_q3, 2), np.percentile(dem_q3, 98)

    def make_rgba(fmap):
        flood_full = np.zeros((H, W), dtype=np.float32)
        for i in range(n_h):
            for j in range(n_w):
                r0,r1 = int(i*ph), int((i+1)*ph)
                c0,c1 = int(j*pw), int((j+1)*pw)
                flood_full[r0:r1, c0:c1] = fmap[i, j]
        rgba = np.zeros((H, W, 4), dtype=np.float32)
        for cls, info in FLOOD_CLASSES.items():
            mask = flood_full == cls
            r,g,b = hex_to_rgb(info['color'])
            rgba[mask,0]=r; rgba[mask,1]=g; rgba[mask,2]=b
            rgba[mask,3] = class_alphas[cls]
        return rgba

    configs = [
        (0, 0, best_idx,  'gt',   'Best  GT'),
        (0, 1, best_idx,  'pred', 'Best  Predicted'),
        (1, 0, worst_idx, 'gt',   'Worst GT'),
        (1, 1, worst_idx, 'pred', 'Worst Predicted'),
    ]

    for row, col, s_idx, mode, label in configs:
        ax = axes[row, col]
        fmap = results['gt_maps'][s_idx] if mode == 'gt' else results['pred_maps'][s_idx]
        m    = results['per_scenario'][s_idx]
        rgba = make_rgba(fmap)

        im = ax.imshow(dem_q3, cmap='terrain', vmin=dem_p2, vmax=dem_p98, origin='upper', interpolation='bilinear')
        ax.imshow(rgba, origin='upper', interpolation='nearest')

        title = (f"RS{m['scenario_id']} — {label}\n"
                 f"Acc={m['accuracy']:.3f}  F1={m['f1_macro']:.3f}  "
                 f"IoU={m['iou_macro']:.3f}")
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_xlabel("Column", fontsize=8)
        ax.set_ylabel("Row", fontsize=8)
        ax.tick_params(labelsize=7)

    # Shared colorbar
    cbar = fig.colorbar(im, ax=axes, fraction=0.015, pad=0.02)
    cbar.set_label("Elevation (m)", fontsize=9)

    # Legend
    patches = [mpatches.Patch(color=FLOOD_CLASSES[c]['color'] if c>0 else '#DDD',
                               label=FLOOD_CLASSES[c]['name']) for c in range(5)]
    fig.legend(handles=patches, loc='lower center', ncol=5,
               fontsize=9, bbox_to_anchor=(0.5, -0.04))
    plt.tight_layout()
    return fig

def plot_confidence_on_dem(
    dem_q3      : np.ndarray,
    conf_map    : np.ndarray,  
    scenario_id : int,
    config_label: str,
    confs_flat  : np.ndarray,   
    figsize     : tuple = (9, 11),
    inset_center: tuple = None,
    inset_size  : int   = 60,
) -> plt.Figure:
    H, W     = dem_q3.shape
    n_h, n_w = conf_map.shape

    # ── Upsample confidence map to DEM resolution ─────────────────────────────
    ph = H / n_h
    pw = W / n_w
    conf_full = np.zeros((H, W), dtype=np.float32)
    for i in range(n_h):
        for j in range(n_w):
            r0, r1 = int(i * ph), int((i+1) * ph)
            c0, c1 = int(j * pw), int((j+1) * pw)
            conf_full[r0:r1, c0:c1] = conf_map[i, j]

    # ── Inset center: default to region of lowest confidence ─────────────────
    if inset_center is None:
        low_patches = np.argwhere(conf_map < 0.5)
        if len(low_patches) > 0:
            pc = low_patches[len(low_patches) // 2]
            inset_center = (int(pc[0] * ph + ph/2), int(pc[1] * pw + pw/2))
        else:
            inset_center = (H//2, W//2)

    ic_r, ic_c = inset_center
    iz  = inset_size
    ir0 = max(0, ic_r - iz); ir1 = min(H, ic_r + iz)
    ic0 = max(0, ic_c - iz); ic1 = min(W, ic_c + iz)

    dem_p2, dem_p98 = np.percentile(dem_q3, 2), np.percentile(dem_q3, 98)
    frac_low        = (confs_flat < 0.5).mean() * 100

    fig = plt.figure(figsize=figsize, facecolor='#1a1a1a')
    ax  = fig.add_axes([0.08, 0.08, 0.72, 0.84], facecolor='#1a1a1a')

    # Layer 1: DEM
    dem_im = ax.imshow(dem_q3, cmap='terrain', vmin=dem_p2, vmax=dem_p98,
                       origin='upper', interpolation='bilinear', alpha=0.5)

    # Layer 2: Confidence heatmap
    conf_im = ax.imshow(conf_full, cmap='RdYlGn', vmin=0.0, vmax=1.0,
                        origin='upper', interpolation='nearest', alpha=0.65)

    # Layer 3: Uncertain patch overlay (conf < 0.5)
    uncertain = conf_full.copy()
    uncertain[conf_full >= 0.5] = np.nan
    uncertain[conf_full <  0.5] = 1.0
    ax.imshow(uncertain, cmap='cool', vmin=0, vmax=1, alpha=0.35,
              origin='upper', interpolation='nearest')

    # Inset marker
    from matplotlib.patches import Rectangle
    rect = Rectangle((ic0, ir0), ic1-ic0, ir1-ir0,
                     linewidth=2, edgecolor='red', facecolor='none', zorder=5)
    ax.add_patch(rect)

    # Axes styling
    ax.set_xlabel("Column (West  ->  East)", fontsize=9, color='white')
    ax.set_ylabel("Row (North  ->  South)",  fontsize=9, color='white')
    ax.tick_params(labelsize=8, colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('white')

    title = (f"Metro Manila Q3 — Model Confidence  (RS{scenario_id})\n"
             f"{config_label}\n"
             f"Mean={confs_flat.mean():.3f}  Std={confs_flat.std():.3f}  "
             f"Min={confs_flat.min():.3f}  Uncertain={frac_low:.1f}%")
    ax.set_title(title, fontsize=10, fontweight='bold', pad=10, color='white')

    # ── DEM colorbar ──────────────────────────────────────────────────────────
    cbar_ax1 = fig.add_axes([0.82, 0.55, 0.025, 0.37])
    cbar1    = fig.colorbar(dem_im, cax=cbar_ax1)
    cbar1.set_label("Elevation (m)", fontsize=8, color='white')
    cbar1.ax.tick_params(labelsize=7, colors='white')
    plt.setp(cbar1.ax.yaxis.get_ticklabels(), color='white')

    # ── Confidence colorbar ───────────────────────────────────────────────────
    cbar_ax2 = fig.add_axes([0.82, 0.10, 0.025, 0.37])
    cbar2    = fig.colorbar(conf_im, cax=cbar_ax2)
    cbar2.set_label("Confidence", fontsize=8, color='white')
    cbar2.set_ticks([0.0, 0.25, 0.5, 0.75, 1.0])
    cbar2.ax.tick_params(labelsize=7, colors='white')
    plt.setp(cbar2.ax.yaxis.get_ticklabels(), color='white')
    cbar2.ax.axhline(y=0.5, color='white', linewidth=1.2, linestyle='--')
    cbar2.ax.text(2.5, 0.5, 'uncertain\nthreshold',
                  fontsize=6, va='center', color='white')

    # ── Inset zoom panel ──────────────────────────────────────────────────────
    inset_ax = fig.add_axes([0.44, 0.08, 0.28, 0.28], facecolor='#1a1a1a')
    inset_ax.imshow(dem_q3[ir0:ir1, ic0:ic1], cmap='terrain',
                    vmin=dem_p2, vmax=dem_p98, origin='upper',
                    interpolation='bilinear', alpha=0.5)
    inset_ax.imshow(conf_full[ir0:ir1, ic0:ic1], cmap='RdYlGn',
                    vmin=0.0, vmax=1.0, origin='upper',
                    interpolation='nearest', alpha=0.65)
    unc_inset = uncertain[ir0:ir1, ic0:ic1]
    inset_ax.imshow(unc_inset, cmap='cool', alpha=0.35,
                    vmin=0, vmax=1, origin='upper', interpolation='nearest')
    inset_ax.set_xticks([]); inset_ax.set_yticks([])
    for spine in inset_ax.spines.values():
        spine.set_edgecolor('red')
        spine.set_linewidth(2)

    return fig

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# from testing import ( prepare_ground_truth, evaluate_all_scenarios, plot_metrics_summary, plot_per_scenario_metrics,plot_per_class_metrics, plot_flood_on_dem )

print("\nPreparing ground truth maps for test scenarios...")
ground_truth = prepare_ground_truth(
    y_test           = dataset['y_test'],     
    quadrant_indices = quadrant_indices,
    patch_metadata   = patch_metadata[0],
)

In [ ]:
print("\nEvaluating all test scenarios...")
results = evaluate_all_scenarios(
    model           = model,
    X_test          = dataset['X_test'],
    y_test          = dataset['y_test'],
    conditioning_vectors = conditioning_vectors,
    ground_truth    = ground_truth,
    batch_size      = cfg.train.batch_size,
    device          = device,
    num_classes     = cfg.model.num_classes,
)

In [ ]:
print("\nGenerating visualizations...")

fig1 = plot_metrics_summary(results)
fig1.savefig("eval_metrics_summary.png", dpi=150, bbox_inches='tight')

fig2 = plot_per_scenario_metrics(results)
fig2.savefig("eval_per_scenario.png", dpi=150, bbox_inches='tight')

fig3 = plot_per_class_metrics(results)
fig3.savefig("eval_per_class.png", dpi=150, bbox_inches='tight')

plt.show()

In [ ]:
H_dem, W_dem = dem_clean.shape
dem_q3 = dem_clean[H_dem//2:, :W_dem//2]   

best_idx  = results['aggregate']['best_scenario']  - 1
worst_idx = results['aggregate']['worst_scenario'] - 1

# Single scenario — predicted
fig4 = plot_flood_on_dem(
    dem_q3      = dem_q3,
    flood_map   = results['pred_maps'][best_idx],
    scenario_id = results['aggregate']['best_scenario'],
    metrics     = results['per_scenario'][best_idx],
    mode        = 'pred',
    inset_center= (150, 420),
    figsize     = (9, 11),
    inset_size  = 60
)
fig4.savefig("eval_dem_best_pred.png", dpi=150, bbox_inches='tight')

# Single scenario — ground truth
fig5 = plot_flood_on_dem(
    dem_q3      = dem_q3,
    flood_map   = results['gt_maps'][best_idx],
    scenario_id = results['aggregate']['best_scenario'],
    metrics     = results['per_scenario'][best_idx],
    mode        = 'gt',
    inset_center=(150, 420),
    figsize     = (9, 11),
    inset_size  = 60
)
fig5.savefig("eval_dem_best_gt.png", dpi=150, bbox_inches='tight')

plt.show()
print("\nAll figures saved.")
print(f"  Best  scenario: RS{results['aggregate']['best_scenario']}")
print(f"  Worst scenario: RS{results['aggregate']['worst_scenario']}")

In [ ]:
fig6 = plot_flood_on_dem(
    dem_q3      = dem_q3,
    flood_map   = results['pred_maps'][worst_idx],
    scenario_id = results['aggregate']['worst_scenario'],
    metrics     = results['per_scenario'][worst_idx],
    mode        = 'pred',
    inset_center= (150, 420),
    figsize     = (9, 11),
    inset_size  = 60
)
fig6.savefig("eval_dem_worst_pred.png", dpi=150, bbox_inches='tight')

# Single scenario — ground truth
fig7 = plot_flood_on_dem(
    dem_q3      = dem_q3,
    flood_map   = results['gt_maps'][worst_idx],
    scenario_id = results['aggregate']['worst_scenario'],
    metrics     = results['per_scenario'][worst_idx],
    mode        = 'gt',
    inset_center=(150, 420),
    figsize     = (9, 11),
    inset_size  = 60
)
fig7.savefig("eval_dem_worst_gt.png", dpi=150, bbox_inches='tight')

In [ ]:
best_idx  = results['aggregate']['best_scenario']  - 1

fig_conf = plot_confidence_on_dem(
    dem_q3       = dem_q3,
    conf_map     = results['conf_maps'][best_idx],
    scenario_id  = results['aggregate']['best_scenario'],
    config_label = 'Full Inputs',
    confs_flat   = results['per_scenario'][best_idx]['confs_flat'],
    inset_center = (150, 420),
    figsize      = (9, 11),
    inset_size   = 60,
)
fig_conf.savefig("eval_dem_best_confidence.png", dpi=150, bbox_inches='tight')
plt.show()

# Web Config

In [ ]:
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import numpy as np
from inference import FloodInference

# ── Visualization Setup ───────────────────────────────────────────────────────
FLOOD_COLORS = {0: '#FFFFFF', 1: '#FFEB3B', 2: '#FF9800', 3: '#F44336', 4: '#9C27B0'}
FLOOD_LABELS = {0: 'No Flood', 1: 'Light', 2: 'Moderate', 3: 'Heavy', 4: 'Extreme'}

flood_cmap = mcolors.ListedColormap([FLOOD_COLORS[i] for i in range(5)])
flood_norm = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], ncolors=5)
conf_cmap = 'RdYlGn'

legend_patches = [
    mpatches.Patch(facecolor=FLOOD_COLORS[i], edgecolor='gray', linewidth=0.5,
                   label=f'Class {i} — {FLOOD_LABELS[i]}')
    for i in range(5)
]

# ── Extract Full Spatial Patches ──────────────────────────────────────────────
print("Extracting full spatial patches...")
X_full = extract_spatial_patches(spatial_stack, patch_size=4)
print(f"  Shape: {X_full.shape}  (85986 patches, 4×4 pixels, 3 channels)")

# ── Initialize Inference Engine ───────────────────────────────────────────────
engine = FloodInference(
    model=model,
    device=device,
    rain_min=rain_min,
    rain_max=rain_max,
    X_spatial=X_full,       
    num_classes=cfg.train.num_classes,         
    batch_size=cfg.train.batch_size,
)

# ── Test Configurations ───────────────────────────────────────────────────────
test_configs = [
    {'storm_type': 'triangular', 'depth_mm': 9, 'tpeak': 0.5},
    {'storm_type': 'front-loaded', 'depth_mm': 9, 'tpeak': None},
    {'storm_type': 'back-loaded', 'depth_mm': 10, 'tpeak': None},
    {'storm_type': 'balanced', 'depth_mm': 40, 'tpeak': None},
]

# ── Run Batch Predictions ─────────────────────────────────────────────────────
# Pass n_h and n_w to predict_batch() to get 2D map reconstruction
results = engine.predict_batch(
    test_configs,
    n_h=306,   # Full domain: 306 rows
    n_w=281,   # Full domain: 281 columns
)

# ── Helper Functions ──────────────────────────────────────────────────────────

def build_subtitle(cfg: dict) -> str:
    """Build informative subtitle."""
    parts = [f"Storm: {cfg['storm_type']}", f"Depth: {cfg['depth_mm']} mm"]
    if cfg.get('tpeak') is not None:
        parts.append(f"tpeak: {cfg['tpeak']}")
    return '  |  '.join(parts)


def add_warning_banner(ax, warnings):
    """Add warning banner if validation warnings exist."""
    if warnings:
        ax.text(
            0.5, 0.02,
            f'⚠ {len(warnings)} warning(s) — inputs outside training range',
            transform=ax.transAxes, ha='center', va='bottom',
            fontsize=7.5, color='white',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#B8860B',
                      edgecolor='none', alpha=0.88),
        )


def compute_class_pcts(flood_map):
    """Compute percentage of each flood class."""
    total = flood_map.size
    return {i: 100 * (flood_map == i).sum() / total for i in range(5)}


def save_result_as_tif(
    flood_map: np.ndarray,
    conf_map: np.ndarray,
    config: dict,
    save_path: str,
    transform=None,
    crs=None,
) -> None:
    """Save prediction as GeoTIFF with metadata."""
    H, W = flood_map.shape
    
    with rasterio.open(
        save_path,
        mode='w',
        driver='GTiff',
        height=H,
        width=W,
        count=2,
        dtype='float32',
        crs=crs,
        transform=transform,
        compress='lzw',
    ) as dst:
        dst.write(flood_map.astype(np.float32), 1)
        dst.write(conf_map.astype(np.float32), 2)
        dst.update_tags(1, description='Flood Hazard Class (0=None 1=Light 2=Moderate 3=Heavy 4=Extreme)')
        dst.update_tags(2, description='Model Confidence (softmax max-probability per patch 0.0-1.0)')
        dst.update_tags(
            model='EXP-DES-2 ViTFloodClassifier',
            domain='Full spatial domain (306×281 patches)',
            storm_type=config['storm_type'],
            depth_mm=str(config['depth_mm']),
            tpeak=str(config.get('tpeak')),
        )
    print(f"  ✓ GeoTIFF saved → {save_path}")


# ── Process Each Result ───────────────────────────────────────────────────────

for i, (result, config) in enumerate(zip(results, test_configs)):
    cfg_label = f"Config {i+1}: {config['storm_type'].title()} | {config['depth_mm']}mm"
    
    # ── Check for errors ──────────────────────────────────────────────────────
    if 'error' in result:
        print(f"\n{cfg_label}: ERROR — {result['error']}")
        continue
    
    # ── Extract data from result ──────────────────────────────────────────────
    print(f"\n{cfg_label}: SUCCESS")
    for cls, stat in result['statistics'].items():
        print(f"  {cls}: {stat['percentage']:.1f}%")
    
    flood_map = result['flood_map'].astype(np.float32)  # (306, 281)
    conf_map = result['conf_map'].astype(np.float32)    # (306, 281)
    patch_confs = result['patch_confidences']           # (85986,)
    warnings = result.get('warnings', [])
    pcts = compute_class_pcts(flood_map)
    frac_low = (patch_confs < 0.5).mean() * 100
    subtitle = build_subtitle(config)
    
    # ── FIGURE 1: Flood Prediction Map ────────────────────────────────────────
    fig1, axes1 = plt.subplots(
        1, 2, figsize=(14, 6),
        gridspec_kw={'width_ratios': [2.5, 1], 'wspace': 0.25}
    )
    ax_flood, ax_bar = axes1
    
    # Plot flood map
    ax_flood.imshow(flood_map, cmap=flood_cmap, norm=flood_norm,
                    interpolation='nearest', origin='upper')
    ax_flood.set_title(f'{cfg_label}\n{subtitle}',
                       fontsize=9, fontweight='bold', pad=5)
    ax_flood.set_xlabel('Column (West → East)', fontsize=8)
    ax_flood.set_ylabel('Row (North → South)', fontsize=8)
    ax_flood.tick_params(labelsize=7)
    add_warning_banner(ax_flood, warnings)
    
    # Class distribution bar chart
    labels = [FLOOD_LABELS[j] for j in range(5)]
    values = [pcts[j] for j in range(5)]
    colors = [FLOOD_COLORS[j] for j in range(5)]
    bars = ax_bar.barh(labels, values, color=colors,
                       edgecolor='gray', linewidth=0.5, height=0.6)
    
    # Add percentage labels on bars
    for bar, val in zip(bars, values):
        if val > 1.5:
            ax_bar.text(bar.get_width() - 0.5, bar.get_y() + bar.get_height() / 2,
                        f'{val:.1f}%', va='center', ha='right',
                        fontsize=8, color='white', fontweight='bold')
        elif val > 0.1:
            ax_bar.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                        f'{val:.1f}%', va='center', ha='left',
                        fontsize=8, color='#333333')
    
    ax_bar.set_xlim(0, 100)
    ax_bar.set_xlabel('Coverage (%)', fontsize=8)
    ax_bar.set_title('Class\nDistribution', fontsize=8.5, fontweight='bold', pad=5)
    ax_bar.tick_params(axis='both', labelsize=8)
    ax_bar.spines['top'].set_visible(False)
    ax_bar.spines['right'].set_visible(False)
    ax_bar.invert_yaxis()
    
    fig1.legend(handles=legend_patches, loc='lower center', ncol=5, fontsize=8.5,
                frameon=True, title='Flood Severity Classes', title_fontsize=8.5,
                bbox_to_anchor=(0.5, -0.05))
    fig1.suptitle(f'EXP-DES-2 — Flood Prediction Map (Full Domain: 306×281)\n{subtitle}',
                  fontsize=11, fontweight='bold', y=0.98)
    
    save_flood = str(cfg.train.output_dir / f'web_config{i+1}_flood_map.png')
    plt.savefig(save_flood, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"  ✓ Flood map saved  → {save_flood}")
    
    # ── FIGURE 2: Confidence Map ──────────────────────────────────────────────
    fig2, axes2 = plt.subplots(
        1, 2, figsize=(14, 6),
        gridspec_kw={'width_ratios': [2.5, 1], 'wspace': 0.30},
    )
    ax_conf, ax_hist = axes2
    
    # Plot confidence map
    im = ax_conf.imshow(conf_map, cmap=conf_cmap, vmin=0.0, vmax=1.0,
                        interpolation='nearest', origin='upper')
    ax_conf.set_title(
        f'{cfg_label}\n{subtitle}\nModel Confidence (softmax max-probability per patch)',
        fontsize=8.5, fontweight='bold', pad=5,
    )
    ax_conf.set_xlabel('Column (West → East)', fontsize=8)
    ax_conf.set_ylabel('Row (North → South)', fontsize=8)
    ax_conf.tick_params(labelsize=7)
    add_warning_banner(ax_conf, warnings)
    
    # Colorbar
    cbar = fig2.colorbar(im, ax=ax_conf, fraction=0.035, pad=0.03)
    cbar.set_label('Confidence', fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    cbar.set_ticks([0.0, 0.25, 0.5, 0.75, 1.0])
    cbar.ax.axhline(y=0.5, color='black', linewidth=1.2, linestyle='--')
    cbar.ax.text(2.3, 0.5, 'uncertain\nthreshold', fontsize=6, va='center',
                 color='black')
    
    # Overlay uncertain regions
    uncertain_overlay = conf_map.copy().astype(float)
    uncertain_overlay[conf_map >= 0.5] = np.nan
    uncertain_overlay[conf_map < 0.5] = 1.0
    ax_conf.imshow(uncertain_overlay, cmap='cool', alpha=0.30,
                   vmin=0, vmax=1, interpolation='nearest', origin='upper')
    
    # Uncertainty stats on map
    ax_conf.text(
        0.02, 0.98, f'{frac_low:.1f}% uncertain\n(conf < 0.5)',
        transform=ax_conf.transAxes, ha='left', va='top', fontsize=7.5,
        color='red' if frac_low > 20 else 'gray',
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                  edgecolor='lightgray', alpha=0.85),
    )
    
    # Confidence histogram
    ax_hist.hist(patch_confs, bins=20, range=(0, 1),
                 color='steelblue', edgecolor='white', linewidth=0.4, alpha=0.85)
    ax_hist.axvline(x=0.5, color='red', linewidth=1.2, linestyle='--',
                    label='Uncertain (0.5)')
    ax_hist.axvline(x=patch_confs.mean(), color='orange', linewidth=1.2,
                    linestyle='-', label=f"Mean ({patch_confs.mean():.2f})")
    ax_hist.set_xlabel('Confidence', fontsize=8)
    ax_hist.set_ylabel('Patches', fontsize=8)
    ax_hist.set_title('Confidence\nDistribution', fontsize=8.5, fontweight='bold', pad=5)
    ax_hist.tick_params(labelsize=7)
    ax_hist.legend(fontsize=6.5, loc='upper left')
    ax_hist.spines['top'].set_visible(False)
    ax_hist.spines['right'].set_visible(False)
    ax_hist.text(
        0.97, 0.97, f'{frac_low:.1f}%\nuncertain',
        transform=ax_hist.transAxes, ha='right', va='top', fontsize=7.5,
        color='red' if frac_low > 20 else 'gray',
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                  edgecolor='lightgray', alpha=0.85),
    )
    
    fig2.suptitle(f'EXP-DES-2 — Model Confidence Map (Full Domain: 306×281)\n{subtitle}',
                  fontsize=11, fontweight='bold', y=0.98)
    
    save_conf = str(cfg.train.output_dir / f'web_config{i+1}_confidence_map.png')
    plt.savefig(save_conf, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"  ✓ Confidence map saved → {save_conf}")
    
    # ── GeoTIFF Export ────────────────────────────────────────────────────────
    try:
        with rasterio.open(dem_path) as src:
            dem_meta = src.meta
        save_result_as_tif(
            flood_map=flood_map,
            conf_map=conf_map,
            config=config,
            save_path=str(cfg.train.output_dir / f'web_config{i+1}_result.tif'),
            transform=dem_meta['transform'],
            crs=dem_meta['crs'],
        )
    except Exception as e:
        print(f"  ℹ Skipped GeoTIFF export: {e}")

print("\n✓ All web configurations processed (full spatial domain).")